In [3]:
import numpy as np
import matplotlib.pyplot as plt
import mne
from mne.preprocessing import compute_proj_ecg
from mne_connectivity import envelope_correlation
import pandas as pd
from scipy.fft import fft,ifft
from sklearn.decomposition import PCA
from sklearn import svm
import pywt
from scipy.signal import stft
from scipy.signal import ShortTimeFFT
import os
from scipy.stats import kurtosis, skew, zscore
from scipy import fft
import emd
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold, LeaveOneGroupOut
#%matplotlib qt

In [4]:
def load_file(file, dataset='tusz'):
    #Loads a single EDF file using MNE
    #Parameters:
    #file: filename for the file WITHOUT extension as a string
    #dataset: name of the dataset, either 'tusz' (default) or 'chb-mit'
    #Returns:
    #raw: raw data in MNE "raw" data structure
    #annot: annotations CSV as a Pandas dataframe. If dataset='chb-mit', annot=None
    #montage: Name of the montage used, None if dataset='chb-mit'
    raw= mne.io.read_raw_edf(file+'.edf', preload=True, verbose='warning')
    if dataset=='chb-mit':
        return raw, None, None
    annot= pd.read_csv(file+'.csv', skiprows=5)
    if dataset=='tusz':
        if 'tcp_ar_a' in file:
            montage= 'tcp_ar_a'
        elif 'tcp_le_a' in file:
            montage= 'tcp_le_a'
        elif 'tcp_ar' in file:
            montage= 'tcp_ar'
        elif 'tcp_le' in file:
            montage= 'tcp_le'
    return raw, annot, montage

In [5]:
def add_annotations(edf, annot_ch, annot):
    events=np.zeros((len(annot_ch),edf.n_times))
    for i, row in annot.iterrows():
        if annot['channel'][i] not in annot_ch:
            continue
        start= edf.time_as_index(annot['start_time'][i])[0] #change to bipolar instead of raw?
        stop= edf.time_as_index(annot['stop_time'][i])[0]
        #print(start)
        #print(stop)
        #print(annot['label'][i])
        if annot['label'][i]=='bckg':
            continue
        elif annot['label'][i]=='cpsz' or annot['label'][i]=='seiz' or annot['label'][i]=='fnsz' or annot['label'][i]=='gnsz' or annot['label'][i]=='spsz' or annot['label'][i]=='absz' or annot['label'][i]=='tnsz' or annot['label'][i]=='cnsz' or annot['label'][i]=='tcsz' or annot['label'][i]=='atsz' or annot['label'][i]=='mysz' or annot['label'][i]=='nesz':
            #print(annot['channel'][i])
            #print(annot_ch)
            j=np.where(annot_ch == annot['channel'][i])[0][0]
            np.ndarray.fill(events[j,start:stop],1)
            continue
        else:
            j=np.where(annot_ch == annot['channel'][i])[0]
            np.ndarray.fill(events[j,start:stop],0)
            print(annot['label'][i])

In [6]:
def get_events(edf, annot_ch=None, annot=None, dataset='tusz', filename= None):
    #Finds seizure events and creates array based on their timings
    #Parameters:
    #edf: an edf file as an MNE "raw" object
    #annot_ch: PD dataframe listing the channels to use in finding events, not used if dataset='chb-mit'
    #annot: PD dataframe containing all annotations from the CSV file, not used if dataset='chb-mit'
    #dataset: name of the dataset used, either 'tusz' (default) or 'chb-mit'
    #filename: filename for the file WITHOUT extension as a string. Only required for CHB-MIT
    #
    #Returns:
    #events: numpy array of events in all channels of size [channels, timepoints]

    if dataset=='chb-mit':
        #For CHB-MIT dataset, seizure_summary file contains start and stop times for each file
        #'events' array gets filled between start and stop times for all channels
        
        seizures= pd.read_csv('physionet.org/files/chbmit/1.0.0/seizure_summary.csv')
        seizure_files= seizures['File_name']
        filename= filename[-8:] #Removes directories from filename if they are present
        i=seizure_files[seizure_files==(filename+'.edf')].index
        #print(i)
        events=np.zeros((len(edf.ch_names),edf.n_times))
        if not i.empty: #check syntax
            i=i[0]
            start=edf.time_as_index(seizures['Seizure_start'][i])[0]
            stop=edf.time_as_index(seizures['Seizure_stop'][i])[0]
            np.ndarray.fill(events[:,int(start):int(stop)],1)
    else:
        #For TUSZ dataset, annotation files contain events, which are channel specific
        #If the channel is in the list of channels we are using, events array is updated accordingly
        
        #annot_ch= pd.unique(annot['channel'])
        events=np.zeros((len(annot_ch),edf.n_times))
        for i, row in annot.iterrows():
            if annot['channel'][i] not in annot_ch:
                continue
            start= edf.time_as_index(annot['start_time'][i])[0] #change to bipolar instead of raw?
            stop= edf.time_as_index(annot['stop_time'][i])[0]
            #print(start)
            #print(stop)
            #print(annot['label'][i])
            if annot['label'][i]=='bckg':
                continue
            elif annot['label'][i]=='cpsz' or annot['label'][i]=='seiz' or annot['label'][i]=='fnsz' or annot['label'][i]=='gnsz' or annot['label'][i]=='spsz' or annot['label'][i]=='absz' or annot['label'][i]=='tnsz' or annot['label'][i]=='cnsz' or annot['label'][i]=='tcsz' or annot['label'][i]=='atsz' or annot['label'][i]=='mysz' or annot['label'][i]=='nesz':
                #print(annot['channel'][i])
                #print(annot_ch)
                j=np.where(annot_ch == annot['channel'][i])[0][0]
                np.ndarray.fill(events[j,start:stop],1)
                continue
            else:
                #There are some events which are neither background nor seizure
                #Print-debugging used to detect where these events occur
                #When testing, no events have been found
                
                #j=np.where(annot_ch == annot['channel'][i])[0]
                #np.ndarray.fill(events[j,start:stop],0)
                print('Non-seizure event')
                print(annot['label'][i])
    return events

In [7]:
def get_channels(montage, mode='single-channel'):
    #Gets list of channels for a given montage. Only used for TUSZ dataset
    #Parameters:
    #montage: the montage used as a string, either 'tcp_ar', 'tcp_le', 'tcp_ar_a', or 'tcp_le_a'
    #mode: either 'single-channel' or 'multi-channel'
    #Returns:
    #channels: numpy array of strings containing bipolar channel names
    if (montage=='tcp_ar' or montage=='tcp_le') and mode=='single-channel':
        channels=np.array(['FP1-F7', 'F7-T3', 'T3-T5', 'T5-O1', 'FP2-F8', 'F8-T4', 'T4-T6', 'T6-O2', 'A1-T3', 'T3-C3', 'C3-CZ',
                  'CZ-C4', 'C4-T4', 'T4-A2', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2'])
    else:
        channels=np.array(['FP1-F7', 'F7-T3', 'T3-T5', 'T5-O1', 'FP2-F8', 'F8-T4', 'T4-T6', 'T6-O2', 'T3-C3', 'C3-CZ',
                  'CZ-C4', 'C4-T4', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2'])
    return channels

In [8]:
def preprocess_data(raw, annot=None, duration=30, dataset='tusz', filename=None, montage='tcp_ar', mode='single-channel',
                   overlap=0, remove_noise=True, test=False, one_channel=None, return_epochs=False):
    # Gets bipolar channels if needed, applies preprocessing filters and resampling, and cuts data into epochs (may move last part to a separate function)
    # Parameters:
    # Raw: edf file loaded into MNE
    # Annot: annotations from CSV loaded into a Pandas dataframe. Only required for TUSZ
    # Duration: Length of epochs in seconds
    # Dataset: either 'tusz' or 'chb-mit'
    # filename: filename WITHOUT extension, as a string. Only required for CHB-MIT
    # Montage: montage used as a string. Only required for TUSZ
    # Mode: 'single-channel' or 'multi-channel'
    # Overlap: amount of overlap between epochs, in seconds.
    # remove_noise: Boolean that determines whether to use filtering to remove line noise at 60 Hz, for testing purposes
    # test: Boolean created for testing purposes to allow returning the EDF file.
    # Returns:
    # epochs_array: Array of shape (num_files, num_epochs, epoch_length*fs) containing data from epochs
    # epoch_sort: 
    
    if dataset=='chb-mit':
        bipolar=raw
        good_channels= ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FZ-CZ', 'CZ-PZ', 'FP2-F4', 'F4-C4',
           'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2', 'P7-T7', 'T7-FT9', 'FT9-FT10', 'FT10-T8', 'T8-P8-1']
        channels= bipolar.ch_names
        for ch in channels:
            if ch not in good_channels:
                bipolar.drop_channels(ch)
        print(bipolar.ch_names)
        annot_ch=None
    else:
        if one_channel is not None:
            annot_ch= np.array([one_channel])
        else:
            annot_ch= get_channels(montage, mode)
        #annot_ch= pd.unique(annot['channel'])
        bipolar=raw.copy()
        skipped_channels=[]
        for ch_bi in annot_ch:
            #print(ch_bi)
            anode, cathode= ch_bi.split('-')
            anode_found= False
            cathode_found= False
            for ch in raw.ch_names:
                if anode in ch:
                    anode= ch
                    anode_found= True
                elif cathode in ch:
                    cathode=ch
                    cathode_found= True
            #print(anode)
            #print(cathode)
            if anode_found and cathode_found:
                bipolar=mne.set_bipolar_reference(bipolar, anode, cathode, ch_name= ch_bi, drop_refs=False, verbose='warning')
            else:
                print(ch_bi)
                skipped_channels.append(ch_bi)
        bipolar.drop_channels(raw.ch_names)
        #Need to set montages to account for these "skipped channels"- also needs to account for the multi-channel case where some channels are always skipped
        if skipped_channels:
            for skipped in skipped_channels:
                index= np.where(annot['channel']==skipped)[0][0]
                print(skipped)
                annot.drop(index, axis=0, inplace=True)
                annot_ch.remove(skipped)
    if test:
        return bipolar
    if remove_noise:
        #print('removing noise')
        preprocessed = bipolar.notch_filter(np.arange(60, 124, 60))
        preprocessed= preprocessed.filter(l_freq=0.5, h_freq=100)
        preprocessed = preprocessed.resample(200)
    else:
        print('not removing noise')
        preprocessed= bipolar.resample(200)
    #print(preprocessed.n_times)
    events=get_events(preprocessed, annot_ch, annot=annot, dataset=dataset, filename=filename)
    timepoints= (np.arange(0, len(events[0]), 1))/200
    epochs = mne.make_fixed_length_epochs(preprocessed, duration=duration, preload=False, overlap=overlap)
    if return_epochs:
        return epochs
    #print(events.shape)
    #print(epochs.events.shape)
    epochs_array=epochs.get_data()
    #print(epochs_array.shape)
    num_epochs=epochs_array.shape[0]; epoch_length=epochs_array.shape[2]
    #events_cutoff= events[:,:num_epochs*epoch_length]
    #print(events_cutoff.shape)
    #epoch_events= np.split(events_cutoff, num_epochs, axis=1)
    step= epoch_length-overlap*200
    #print(step)
    epoch_events= [events[:,i:i+epoch_length] for i in range(0, len(events[0]), step) if i+epoch_length<=len(events[0])]
    epoch_timepoints= [timepoints[i:i+epoch_length] for i in range(0, len(events[0]), step) if i+epoch_length<=len(events[0])]
    #print(len(epoch_events))
    #print(epoch_events[0].shape)
    #print(epoch_events)
    epoch_sum= np.sum(epoch_events, axis=2)
    #print(epoch_sum.shape)
    epoch_sort=epoch_sum> epoch_length/2
    epoch_timepoints= np.median(epoch_timepoints, axis=1)
    #print(epoch_sort.shape)
    return epochs_array, epoch_sort, preprocessed, epoch_timepoints, annot_ch

In [9]:
def prep_inputs_new (epochs_array, epoch_sort, predict=30, dataset='tusz', mode='single-channel', fs=200, overlap=0,
                    epoch_timepoints=None):
    num_epochs=epochs_array.shape[0]; epoch_length=epochs_array.shape[2]
    num_channels=epochs_array.shape[1]
    #print(predict)
    #print(epoch_length)
    predict= np.round(predict*fs/(epoch_length-overlap*fs))
    #print(predict)
    predict= int(predict)
    #print(predict)
    if mode=='single-channel':
        y0=np.zeros([epoch_sort.size, 3])
        n=0
        for i, epoch in enumerate(epochs_array):
            for j,ch in enumerate(epoch):
                if epoch_sort[i,j]: 
                    y0[n, 0]=1
                elif i+predict<num_epochs and any(epoch_sort[i:i+predict+1,j]): 
                    y0[n, 1]=1
                elif i+predict>=num_epochs and any(epoch_sort[i:-1, j]):
                    y0[n, 1]=1
                else:
                    y0[n, 2]=1
                #print(n)
                n=n+1
                #print('Channel '+str(j))
                #print('Epoch '+str(i))
        x0=np.reshape(epochs_array, (epochs_array.shape[0]*epochs_array.shape[1],epochs_array.shape[2]))
        if epoch_timepoints is not None:
            epoch_timepoints= np.repeat(epoch_timepoints, num_channels)
    else:
        y0=np.zeros([epoch_sort.shape[0], 3])
        #print(epoch_sort.shape)
        epochs_all_channels= np.sum(epoch_sort, axis=1)
        epoch_sort= epochs_all_channels>=1
        #print(epoch_sort.shape)
        for i, epoch in enumerate(epochs_array):
            if epoch_sort[i]: 
                y0[i, 0]=1
            elif i+predict<num_epochs and epoch_sort[i+predict]: 
                y0[i, 1]=1
            else:
                y0[i, 2]=1
            #print(n)
            #print('Channel '+str(j))
            #print('Epoch '+str(i))
        x0=epochs_array
    x0=x0.astype('float32')
    #Currently fills out y with zeros
    return x0, y0, epoch_timepoints

In [10]:
def get_file_list(dataset='tusz', patient='all'):
    if dataset=='chb-mit':
        root_dir='physionet.org/files/chbmit/1.0.0/'
    else:
        root_dir = 'isip.piconepress.com/projects/tuh_eeg/downloads/tuh_eeg_seizure/v2.0.3/edf/train/'
    if patient !='all':
        root_dir= os.path.join(root_dir+patient+'/')
    file_list= []
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            if ('.edf' in file) and ('.seizure' not in file):
                filename_no_ext= root+'/'+file[:-4]
                file_list=np.append(file_list, filename_no_ext)
    return file_list

In [29]:
def get_metadata(annot, y0, file, patient_IDs, time, channels, num_channels):
    metadata= pd.DataFrame(columns=['Patient', 'File', 'Time', 'Channel', 'Seizure Start', 'Seizure Stop', 'Time Since Seizure'])
    metadata['Patient']=patient_IDs
    metadata['File']= np.repeat(file, len(y0))
    metadata['Time']=time
    metadata['Channel']= np.tile(channels, num_channels)
    start_times=np.array([])
    stop_times=np.array([])
    channel_times= np.array([])
    for i, row in annot.iterrows():
        #print(i)
        #print(row['label'])
        #print(row['start_time'])
        if row['label']=='bckg':
            continue
        else:
            start_times=np.append(start_times,row['start_time'])
            stop_times=np.append(stop_times, row['stop_time'])
            channel_times=np.append(channel_times, row['channel'])
    #print(start_times)
    for channel in channels:
        ch_start_times= start_times[channel_times==channel]
        ch_stop_times= stop_times[channel_times==channel]
        last_i=0
        if len(ch_start_times)==0:
            continue
        for i,ch_start_time in enumerate(ch_start_times):
            if i<len(start_times)-1:
                if i>0:
                    index= metadata.index[(metadata['Time']<=ch_stop_times[i]) & (metadata['Time']>ch_stop_times[i-1])]
                else:
                    index= metadata.index[(metadata['Time']<ch_stop_times[i])]
            else:
                index= metadata.index[last_i:]
            index_ch=metadata.index[metadata['Channel']==channel]
            index= index.intersection(index_ch)
            #print(index)
            metadata.loc[index, 'Seizure Start']= ch_start_time
            metadata.loc[index, 'Seizure Stop']= ch_stop_times[i]
            metadata.loc[index, 'Time Since Seizure']= metadata.loc[index, 'Time']-ch_start_time
            last_i=index[len(index)-1]
    return metadata

In [12]:
def load_all_data(file_list, x=np.array([]), y=np.array([]), skipped_files= np.array([]), patient_IDs= np.array([]), predict=30,
                  duration=30, dataset='tusz', mode='single-channel', overlap=0, remove_noise=True, one_channel=None,
                 print_num_preictal=False):
    #Need code to get list of all possible filenames and iterate through them all
    
    #file_list= np.delete(file_list, np.where('isip.piconepress.com/projects/tuh_eeg/downloads/tuh_eeg_seizure/v2.0.3/edf/train/aaaaalmx/s002_2011/03_tcp_ar_a/aaaaalmx_s002_t000'))
    #file_list= np.delete(file_list, np.where('isip.piconepress.com/projects/tuh_eeg/downloads/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaacz/s006_2015/03_tcp_ar_a/aaaaaacz_s006_t000'))
    #skipped_files= np.array(['isip.piconepress.com/projects/tuh_eeg/downloads/tuh_eeg_seizure/v2.0.3/edf/train/aaaaalmx/s002_2011/03_tcp_ar_a/aaaaalmx_s002_t000',
    #                         'isip.piconepress.com/projects/tuh_eeg/downloads/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaacz/s006_2015/03_tcp_ar_a/aaaaaacz_s006_t000'])
    #skipped_files= np.array([])
    #file_list= np.flip(file_list)
    #x, y= [], []
    #initialized= False
    for file in file_list:
        #print(file)
        patient= file[-18:-10]
        print(patient)
        raw, annot, montage= load_file(file, dataset= dataset)
        if raw.times[-1]<duration:
            skipped_files= np.append(skipped_files, file)
            continue
        epochs_array, epoch_sort, preprocessed, epoch_timepoints, channels= preprocess_data(raw, annot, duration=duration, dataset=dataset, filename=file, montage=montage,
                                                                                  mode=mode, overlap=overlap, remove_noise=remove_noise, 
                                                                                  one_channel=one_channel)
        x0, y0, time= prep_inputs_new(epochs_array, epoch_sort, predict=predict, dataset=dataset, mode=mode, overlap=overlap, epoch_timepoints=epoch_timepoints)
        if print_num_preictal:
            print(file)
            print('Preictal epochs: ')
            print(len(np.where(y0[:,1]==1)[0]))
        if(x0.shape[0]!=y0.shape[0]):
            print(file)
        if x.size==0:
            x= x0
            y= y0
            patient_IDs= np.repeat(patient, len(y0))
            metadata= get_metadata(annot, y0, file, patient_IDs, time, channels, epochs_array.shape[0])
        else:
            x= np.append(x, x0, axis=0)
            y= np.append(y, y0, axis=0)
            patient_IDs= np.repeat(patient, len(y0))
            meta_0= get_metadata(annot, y0, file, patient_IDs, time, channels, epochs_array.shape[0])
            metadata= pd.concat((metadata, meta_0), axis=0, ignore_index=True)
    return x,y, skipped_files, metadata

In [1]:
def truncate_data(x,y, metadata, seeded=False, seed=42):
    metadata= metadata.reset_index()
    index_ictal=np.where(y[:,0]==1)[0]
    index_pre=np.where(y[:,1]==1)[0]
    index_none=np.where(y[:,2]==1)[0]
    if index_pre.size>0:
        length= np.min([len(index_ictal), len(index_pre), len(index_none)])
    #print(min_index)
    else:
        length= np.min([len(index_ictal), len(index_none)])
    #print(length)
    if seeded:
        np.random.seed(seed)
    np.random.shuffle(index_ictal)
    np.random.shuffle(index_none)
    np.random.shuffle(index_pre)
    index_ictal_short= index_ictal[:length]
    index_none_short= index_none[:length]
    if index_pre.size>0:
        index_pre_short= index_pre[:length]
        x_short=np.append(x[index_pre_short], x[index_ictal_short], axis=0)
        x_short=np.append(x_short, x[index_none_short], axis=0)
        y_short=np.append(y[index_pre_short], y[index_ictal_short], axis=0)
        y_short=np.append(y_short, y[index_none_short], axis=0)
        meta_short= pd.concat([metadata.loc[index_pre_short], metadata.loc[index_ictal_short], metadata.loc[index_none_short]], axis=0)
    else:
        x_short=np.append(x[index_ictal_short], x[index_none_short], axis=0)
        y_short=np.append(y[index_ictal_short], y[index_none_short], axis=0)
        meta_short= pd.concat([metadata.loc[index_ictal_short], metadata.loc[index_none_short]], axis=0)
    return x_short, y_short, meta_short

In [14]:
def truncate_data_no_preictal(x,y, patients, seeded=False, seed=42):
    index_ictal=np.where(y[:,0]==1)[0]
    #index_pre=np.where(y[:,1]==1)[0]
    index_none=np.where(y[:,1]==1)[0]
    length= len(index_ictal)
    #index_ictal_short= index_ictal[:length]
    index_none_short= index_none[:length]
    if seeded:
        np.random.seed(seed)
    #np.random.shuffle(index_ictal)
    np.random.shuffle(index_none)
    #x_short=np.append(x[index_pre], x[index_ictal_short], axis=0)
    x_short=np.append(x[index_ictal], x[index_none_short], axis=0)
    #y_short=np.append(y[index_pre], y[index_ictal_short], axis=0)
    y_short=np.append(y[index_ictal], y[index_none_short], axis=0)
    patients_short=np.append(patients[index_ictal], patients[index_none_short], axis=0)
    return x_short, y_short, patients_short

In [15]:
def STFT_transform(x, win_length=200, hop=50, fs=200, reshape=True):
    stft= ShortTimeFFT(win=np.ones(win_length), hop=hop, fs=fs)
    x_stft=stft.spectrogram(x)
    if reshape:
        x_stft=np.reshape(x_stft,(x_stft.shape[0],x_stft.shape[1],x_stft.shape[2],1))
    return x_stft

In [16]:
def CWT_transform(x, wavelet='morl', freqs=np.arange(1,100.5,2), fs=200):
    scale= pywt.frequency2scale(wavelet,freqs/fs)
    coef, freqs=pywt.cwt(x,scale,wavelet,sampling_period=1/fs)
    coef= np.swapaxes(coef, 0, 1)
    return coef, freqs

In [17]:
def PCA_transform(x, num_components=10):
    pca= PCA()
    x_pca=pca.fit_transform(x)
    x_pca= x_pca[:,:num_components]
    return x_pca

In [18]:
def dwt_transform(x, wavelet='db2'):
    cA, cD = pywt.dwt(x, wavelet)
    return cA, cD

In [19]:
def get_splits(x, y, metadata, seeded=False, seed=42, group=False):
    if group:
        if seeded:
            gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
        else:
            gss = GroupShuffleSplit(n_splits=1, test_size=0.2)
        train, test= next(gss.split(x, y, metadata['File']))
        #print(patients[test])
        x_train, x_test= x[train], x[test]
        y_train, y_test= y[train], y[test]
        meta_train, meta_test= metadata.loc[train], metadata.loc[test]
        return x_train, x_test, y_train, y_test, meta_train, meta_test
    elif seeded:
        x_train, x_test, y_train, y_test= train_test_split(x, y, test_size= 0.20, random_state=seed, stratify=y)
    else:
        x_train, x_test, y_train, y_test= train_test_split(x, y, test_size= 0.20, stratify=y)
    return x_train, x_test, y_train, y_test

In [20]:
def get_kfold_splits(x, y, metadata, seeded=False, seed=42, n_splits=5, mode='stratify'):
    if mode=='logo':
        gkf= LeaveOneGroupOut()
    elif mode=='stratify':
        gkf=StratifiedGroupKFold(n_splits=n_splits)
    else:
        gkf=GroupKFold(n_splits=n_splits)
    split= gkf.split(x, y, metadata['File'])
    return split